# SentinelNet v4.0 + Seq2Seq — T4 GPU Training
**Author:** @who_is_the_black_hat

**Upload to Google Drive root:**
- `threat_master_balanced.jsonl`
- `linux_kali_dataset.jsonl`
- `generative_training.jsonl` ← NEW (seq2seq)

**Downloads:**
- `sentinel_threat_net.pt` + `sentinel_vocab.json` (classifier)
- `sentinel_seq2seq.pt` + `sentinel_seq2seq_vocab.json` (seq2seq)

```bash
cp ~/Downloads/sentinel_threat_net.pt    /home/kali/osints/models/ml_engine/
cp ~/Downloads/sentinel_vocab.json       /home/kali/osints/models/ml_engine/
cp ~/Downloads/sentinel_seq2seq.pt       /home/kali/osints/models/ml_engine/
cp ~/Downloads/sentinel_seq2seq_vocab.json /home/kali/osints/models/ml_engine/
```

In [ ]:
# Cell 1 — GPU Check
import torch
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('CUDA :', torch.cuda.is_available())
print('PyTorch:', torch.__version__)

In [ ]:
# Cell 2 — Load Data from Google Drive
from google.colab import drive
from pathlib import Path
import json, glob, random
from collections import Counter, defaultdict

drive.mount('/content/drive', force_remount=True)

THREAT_LABELS = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
LABEL_MAP = {
    'CRITICAL': 'CRITICAL', 'HIGH': 'HIGH',
    'MEDIUM': 'MEDIUM', 'MODERATE': 'MEDIUM',
    'LOW': 'LOW'
}

def load_jsonl(path, max_per_file=60000):
    samples = []
    seen = set()
    with open(path, encoding='utf-8', errors='replace') as f:
        for line in f:
            if len(samples) >= max_per_file:
                break
            try:
                d = json.loads(line.strip())
                text  = str(d.get('text') or d.get('content') or '').strip()
                label = LABEL_MAP.get(str(d.get('label', '')).upper(), '')
                if not text or not label or len(text) < 20:
                    continue
                key = text[:80]
                if key in seen:
                    continue
                seen.add(key)
                samples.append({
                    'text': text[:1500],
                    'label': label,
                    'threat_type': d.get('threat_type', 'unknown'),
                    'action_hint': d.get('action_hint', 'monitor')
                })
            except:
                pass
    return samples

all_samples = []

# Search Drive for data files
files_to_find = [
    'threat_master_balanced.jsonl',
    'linux_kali_dataset.jsonl',
    'threat_v4_final.jsonl',
    'github_threat_data.jsonl',
    'threat_v4_clean.jsonl',
]

for fname in files_to_find:
    found = glob.glob(f'/content/drive/MyDrive/**/{fname}', recursive=True)
    if not found:
        found = glob.glob(f'/content/drive/MyDrive/{fname}')
    if found:
        s = load_jsonl(found[0])
        all_samples.extend(s)
        print(f'✓ {fname}: {len(s):,} samples')
    else:
        print(f'✗ {fname}: not found')

# Balance classes
buckets = defaultdict(list)
for s in all_samples:
    buckets[s['label']].append(s)

print('\nBefore balance:')
for label, items in sorted(buckets.items()):
    print(f'  {label}: {len(items):,}')

MAX_PER_CLASS = 15000
samples = []
for label, items in buckets.items():
    random.shuffle(items)
    samples.extend(items[:MAX_PER_CLASS])
random.shuffle(samples)

print(f'\nAfter balance: {len(samples):,} samples')
print(dict(Counter(s['label'] for s in samples)))

In [ ]:
# Cell 2b — Enrich: threat_type + action_hint auto-assign
# Training data mein ye fields nahi hain — keywords se assign karo

TYPE_KEYWORDS = {
    'recon':      ['nmap','scan','subdomain','whois','dns','recon','enumerat','subfinder','amass','masscan','shodan','port','fingerprint','banner'],
    'web_vuln':   ['xss','sqli','sql injection','sqlmap','nikto','gobuster','ffuf','lfi','rfi','xxe','ssti','cors','csrf','ssrf','idor','injection','payload','burp','nuclei','dirb','dirbuster','wfuzz','endpoint','admin panel','backup'],
    'breach':     ['breach','leak','credential','password','dump','stealer','infostealer','haveibeenpwned','hibp','dehashed','pastebin','combo','stuffing','hash','cracked','hydra','hashcat','john'],
    'malware':    ['ransomware','malware','trojan','backdoor','rootkit','botnet','keylogger','fileless','worm','virus','dropper','loader','rat','c2','command and control','cobalt strike','mimikatz','metasploit','meterpreter','payload','shellcode'],
    'phishing':   ['phishing','spearphishing','email','smtp','spoof','fake','impersonat','social engineer','pretexting','vishing','smishing','lure','attachment','macro'],
    'apt':        ['apt','nation state','advanced persistent','threat actor','campaign','ttps','mitre','lateral movement','exfiltrat','persistence','privilege escalat','zero day','zero-day','supply chain'],
    'misconfig':  ['misconfigur','exposed','public bucket','s3','open port','default password','weak','insecure','header','ssl','tls','certificate','firewall','permission','access control'],
    'exploit':    ['exploit','cve','rce','remote code','buffer overflow','heap','stack','use after free','format string','searchsploit','metasploit','msfvenom','poc','proof of concept'],
}

ACTION_KEYWORDS = {
    'patch_now':        ['cve','rce','remote code','sql injection','sqli','ransomware','zero-day','critical','exploit','buffer overflow'],
    'escalate':         ['apt','nation state','breach','stealer','infostealer','lateral movement','exfiltrat','mimikatz','cobalt strike','active attack'],
    'block_ip':         ['botnet','c2','command and control','malware','ddos','brute force','hydra','scanner','masscan'],
    'investigate':      ['phishing','spearphishing','suspicious','anomaly','unusual','unknown','backdoor','trojan','rat'],
    'collect_evidence': ['forensic','incident','compromise','breach','dump','exfiltrat','log','artifact'],
    'notify_team':      ['critical','high severity','data breach','ransomware','apt','nation state'],
    'monitor':          ['recon','scan','probe','low','info','normal','no vulnerability','ssl grade a'],
}

def assign_type(text):
    t = text.lower()
    scores = {typ: sum(1 for kw in kws if kw in t) for typ, kws in TYPE_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'unknown'

def assign_action(text, label):
    t = text.lower()
    if label == 'CRITICAL':
        for kw in ACTION_KEYWORDS['patch_now']:
            if kw in t: return 'patch_now'
        for kw in ACTION_KEYWORDS['escalate']:
            if kw in t: return 'escalate'
        return 'patch_now'
    elif label == 'HIGH':
        for kw in ACTION_KEYWORDS['block_ip']:
            if kw in t: return 'block_ip'
        for kw in ACTION_KEYWORDS['investigate']:
            if kw in t: return 'investigate'
        return 'patch_now'
    elif label == 'MEDIUM':
        for kw in ACTION_KEYWORDS['collect_evidence']:
            if kw in t: return 'collect_evidence'
        return 'investigate'
    return 'monitor'

# Synthetic samples — tool-specific patterns jo data mein kam hain
SYNTHETIC = [
    # CRITICAL
    {'text': 'sqlmap found sql injection vulnerability database dumped credentials exposed', 'label': 'CRITICAL', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'remote code execution rce vulnerability exploited server compromised', 'label': 'CRITICAL', 'threat_type': 'exploit', 'action_hint': 'patch_now'},
    {'text': 'ransomware encrypted all files bitcoin ransom demand critical infrastructure', 'label': 'CRITICAL', 'threat_type': 'malware', 'action_hint': 'escalate'},
    {'text': 'mimikatz credential dump lsass memory passwords extracted domain admin', 'label': 'CRITICAL', 'threat_type': 'malware', 'action_hint': 'escalate'},
    {'text': 'cobalt strike beacon detected c2 communication lateral movement active', 'label': 'CRITICAL', 'threat_type': 'apt', 'action_hint': 'escalate'},
    {'text': 'zero day exploit used in wild no patch available critical systems affected', 'label': 'CRITICAL', 'threat_type': 'exploit', 'action_hint': 'patch_now'},
    {'text': 'infostealer logs found credentials passwords cookies browser data leaked', 'label': 'CRITICAL', 'threat_type': 'breach', 'action_hint': 'escalate'},
    {'text': 'subdomain takeover vulnerable dangling dns cname pointing expired service', 'label': 'CRITICAL', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'dns zone transfer axfr allowed all dns records exposed nameserver', 'label': 'CRITICAL', 'threat_type': 'misconfig', 'action_hint': 'patch_now'},
    {'text': 'ssti server side template injection jinja2 rce possible template render', 'label': 'CRITICAL', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'lfi local file inclusion etc passwd exposed path traversal vulnerability', 'label': 'CRITICAL', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'xxe xml external entity injection internal file read ssrf possible', 'label': 'CRITICAL', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'metasploit exploit successful meterpreter session opened target compromised', 'label': 'CRITICAL', 'threat_type': 'exploit', 'action_hint': 'escalate'},
    {'text': 'data breach millions records exposed database leaked dark web', 'label': 'CRITICAL', 'threat_type': 'breach', 'action_hint': 'escalate'},
    {'text': 'apt group nation state attack supply chain compromise backdoor inserted', 'label': 'CRITICAL', 'threat_type': 'apt', 'action_hint': 'escalate'},
    # HIGH
    {'text': 'nmap scan found open ports 22 80 443 3306 mysql ssh running services', 'label': 'HIGH', 'threat_type': 'recon', 'action_hint': 'investigate'},
    {'text': 'nikto scan found xss reflected vulnerability admin panel exposed sensitive', 'label': 'HIGH', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'gobuster found admin panel backup files git exposed sensitive data', 'label': 'HIGH', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'nuclei high severity vulnerability found web application outdated version', 'label': 'HIGH', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'hydra brute force attack ssh login multiple failed attempts detected', 'label': 'HIGH', 'threat_type': 'breach', 'action_hint': 'block_ip'},
    {'text': 'cors misconfiguration wildcard origin credentials allowed cross origin', 'label': 'HIGH', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'jwt none algorithm accepted authentication bypass possible token forged', 'label': 'HIGH', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'ssrf server side request forgery internal network access cloud metadata', 'label': 'HIGH', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'github secrets exposed api key token hardcoded in public repository', 'label': 'HIGH', 'threat_type': 'breach', 'action_hint': 'patch_now'},
    {'text': 'phishing email campaign spearphishing attachment malicious macro detected', 'label': 'HIGH', 'threat_type': 'phishing', 'action_hint': 'investigate'},
    {'text': 'open redirect vulnerability url parameter unvalidated redirect external', 'label': 'HIGH', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'clickjacking x-frame-options missing page can be framed ui redress', 'label': 'HIGH', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'searchsploit found exploits for service version running on target', 'label': 'HIGH', 'threat_type': 'exploit', 'action_hint': 'patch_now'},
    {'text': 'cloud storage s3 bucket public read access sensitive files exposed', 'label': 'HIGH', 'threat_type': 'misconfig', 'action_hint': 'patch_now'},
    {'text': 'password hash cracked hashcat john the ripper weak password found', 'label': 'HIGH', 'threat_type': 'breach', 'action_hint': 'investigate'},
    # MEDIUM
    {'text': 'subfinder amass subdomain enumeration 50 subdomains found attack surface', 'label': 'MEDIUM', 'threat_type': 'recon', 'action_hint': 'investigate'},
    {'text': 'security headers missing x-content-type csp hsts not configured', 'label': 'MEDIUM', 'threat_type': 'misconfig', 'action_hint': 'patch_now'},
    {'text': 'ssl tls weak cipher suite detected tls 1.0 enabled downgrade possible', 'label': 'MEDIUM', 'threat_type': 'misconfig', 'action_hint': 'patch_now'},
    {'text': 'directory listing enabled sensitive files visible web server misconfigured', 'label': 'MEDIUM', 'threat_type': 'misconfig', 'action_hint': 'patch_now'},
    {'text': 'whois dns records exposed registrar info nameservers ip addresses', 'label': 'MEDIUM', 'threat_type': 'recon', 'action_hint': 'monitor'},
    {'text': 'wayback machine historical urls parameters found old endpoints exposed', 'label': 'MEDIUM', 'threat_type': 'recon', 'action_hint': 'investigate'},
    {'text': 'cookie missing httponly secure flag session token exposed javascript', 'label': 'MEDIUM', 'threat_type': 'web_vuln', 'action_hint': 'patch_now'},
    {'text': 'theHarvester email addresses found domain employees exposed osint', 'label': 'MEDIUM', 'threat_type': 'recon', 'action_hint': 'monitor'},
    # LOW
    {'text': 'normal website no vulnerabilities found ssl grade A security headers ok', 'label': 'LOW', 'threat_type': 'misconfig', 'action_hint': 'monitor'},
    {'text': 'scan complete no critical findings all security checks passed clean', 'label': 'LOW', 'threat_type': 'unknown', 'action_hint': 'monitor'},
    {'text': 'ssl certificate valid tls 1.3 strong ciphers no vulnerabilities detected', 'label': 'LOW', 'threat_type': 'misconfig', 'action_hint': 'monitor'},
    {'text': 'no open ports found firewall blocking all connections target hardened', 'label': 'LOW', 'threat_type': 'recon', 'action_hint': 'monitor'},
    {'text': 'informational finding low risk no immediate action required monitor', 'label': 'LOW', 'threat_type': 'unknown', 'action_hint': 'monitor'},
]

# 500x augment karo — har synthetic sample 500 baar add karo
for s in SYNTHETIC:
    samples.extend([s.copy() for _ in range(500)])
random.shuffle(samples)
print(f'After synthetic augment: {len(samples):,} samples')
print(dict(Counter(s['label'] for s in samples)))

# Enrich all samples
for s in samples:
    if s.get('threat_type', 'unknown') == 'unknown':
        s['threat_type'] = assign_type(s['text'])
    if s.get('action_hint', 'monitor') == 'monitor':
        s['action_hint'] = assign_action(s['text'], s['label'])

# Label correction — keywords se label override karo agar clearly wrong hai
CRITICAL_KW = ['ransomware','remote code execution','rce','zero-day','zero day','apt','nation state','supply chain','mimikatz','cobalt strike','active exploit','data exfiltrat','infostealer','stealer log']
HIGH_KW     = ['sql injection','sqli','sqlmap','xss','cross-site scripting','lfi','rfi','xxe','ssti','ssrf','command injection','auth bypass','subdomain takeover','zone transfer','nuclei critical','cve-']
LOW_KW      = ['no vulnerability','no vulnerabilities','ssl grade a','headers ok','not vulnerable','no issues found','clean scan','no findings','low risk','informational']

fixed = 0
for s in samples:
    t = s['text'].lower()
    if any(kw in t for kw in CRITICAL_KW) and s['label'] != 'CRITICAL':
        s['label'] = 'CRITICAL'
        fixed += 1
    elif any(kw in t for kw in HIGH_KW) and s['label'] == 'LOW':
        s['label'] = 'HIGH'
        fixed += 1
    elif any(kw in t for kw in LOW_KW) and s['label'] in ('CRITICAL','HIGH'):
        s['label'] = 'LOW'
        fixed += 1

print(f'Labels fixed: {fixed}')

# Re-balance after correction
from collections import Counter, defaultdict
import random
buckets2 = defaultdict(list)
for s in samples: buckets2[s['label']].append(s)
print('After correction:')
for label, items in sorted(buckets2.items()): print(f'  {label}: {len(items):,}')
samples = []
for label, items in buckets2.items():
    random.shuffle(items)
    samples.extend(items[:15000])
random.shuffle(samples)
print(f'Final: {len(samples):,} samples')

from collections import Counter
print('threat_type distribution:')
print(dict(Counter(s['threat_type'] for s in samples)))
print('action_hint distribution:')
print(dict(Counter(s['action_hint'] for s in samples)))

In [ ]:
# Cell 3 — Model Definition
import re, math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

THREAT_LABELS = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
LABEL2IDX = {l: i for i, l in enumerate(THREAT_LABELS)}
IDX2LABEL = {i: l for i, l in enumerate(THREAT_LABELS)}

THREAT_TYPES = ['recon', 'web_vuln', 'breach', 'malware', 'phishing',
                'apt', 'insider', 'misconfig', 'social_eng', 'unknown']
TYPE2IDX = {t: i for i, t in enumerate(THREAT_TYPES)}
IDX2TYPE = {i: t for i, t in enumerate(THREAT_TYPES)}

ACTION_HINTS = ['monitor', 'patch_now', 'block_ip', 'escalate', 'investigate',
                'notify_team', 'collect_evidence', 'no_action']
HINT2IDX = {h: i for i, h in enumerate(ACTION_HINTS)}
IDX2HINT = {i: h for i, h in enumerate(ACTION_HINTS)}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class SentinelTokenizer:
    SECURITY_TERMS = {
        'ransomware','malware','exploit','backdoor','trojan','botnet',
        'phishing','zero-day','apt','c2','sqli','xss','ssrf','rce',
        'lfi','xxe','ssti','cors','jwt','oauth','infostealer','keylogger',
        'rootkit','fileless','mimikatz','cobalt-strike','metasploit',
        'cve','cvss','mitre','osint','exfiltration','persistence',
        'nmap','sqlmap','nikto','nuclei','gobuster','hydra','hashcat',
        'msfconsole','searchsploit','netcat','wireshark','burpsuite',
        'privilege-escalation','lateral-movement','command-and-control',
        'darkweb','tor','onion','kali','pentest','ctf','redteam',
    }
    PAD_IDX = 0
    UNK_IDX = 1

    def __init__(self, max_vocab=15000):
        self.max_vocab = max_vocab
        self.word2idx  = {'<PAD>': 0, '<UNK>': 1}
        self.vocab_size = 2

    def _tokenize(self, text):
        return [w for w in re.findall(r'[a-z0-9]+(?:-[a-z0-9]+)*', text.lower()) if len(w) >= 2]

    def build_vocab(self, texts):
        from collections import Counter
        counter = Counter()
        for t in texts:
            counter.update(self._tokenize(t))
        for term in self.SECURITY_TERMS:
            counter[term] = counter.get(term, 0) + 1000
        for word, _ in counter.most_common(self.max_vocab - 2):
            if word not in self.word2idx:
                self.word2idx[word] = len(self.word2idx)
        self.vocab_size = len(self.word2idx)
        print(f'Vocab: {self.vocab_size} tokens')

    def encode(self, text, max_len=300):
        ids = [self.word2idx.get(t, self.UNK_IDX) for t in self._tokenize(text)[:max_len]]
        return ids + [self.PAD_IDX] * (max_len - len(ids))

    def save(self, path):
        with open(path, 'w') as f:
            json.dump({'word2idx': self.word2idx, 'max_vocab': self.max_vocab}, f)


class ThreatDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=300):
        self.encodings    = [tokenizer.encode(s['text'], max_len) for s in samples]
        self.labels       = [LABEL2IDX[s['label']] for s in samples]
        self.threat_types = [TYPE2IDX.get(s.get('threat_type', 'unknown'), 9) for s in samples]
        self.action_hints = [HINT2IDX.get(s.get('action_hint', 'monitor'), 0) for s in samples]

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.encodings[idx],   dtype=torch.long),
            torch.tensor(self.labels[idx],       dtype=torch.long),
            torch.tensor(self.threat_types[idx], dtype=torch.long),
            torch.tensor(self.action_hints[idx], dtype=torch.long),
        )


class SentinelNet(nn.Module):
    VERSION = '4.0'
    AUTHOR  = 'who_is_the_black_hat'

    def __init__(self, vocab_size, embed_dim=128, num_filters=128,
                 kernels=(3, 5, 7), dropout=0.4, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos_drop  = nn.Dropout(dropout)
        self.convs = nn.ModuleList([
            nn.Sequential(nn.Conv1d(embed_dim, num_filters, k, padding=k//2), nn.GELU())
            for k in kernels
        ])
        cnn_out = num_filters * len(kernels)
        self.cnn_proj  = nn.Linear(cnn_out, embed_dim)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.head_label  = self._head(embed_dim, len(THREAT_LABELS), dropout)
        self.head_type   = self._head(embed_dim, len(THREAT_TYPES),  dropout)
        self.head_action = self._head(embed_dim, len(ACTION_HINTS),  dropout)
        for n, p in self.named_parameters():
            if 'weight' in n and p.dim() >= 2: nn.init.xavier_uniform_(p)
            elif 'bias' in n: nn.init.zeros_(p)

    @staticmethod
    def _head(in_d, out_d, dr):
        return nn.Sequential(
            nn.Dropout(dr), nn.Linear(in_d, in_d//2),
            nn.GELU(), nn.Linear(in_d//2, out_d)
        )

    def forward(self, x):
        emb    = self.pos_drop(self.embedding(x)).transpose(1, 2)
        pooled = torch.cat([c(emb).max(dim=2).values for c in self.convs], dim=-1)
        ctx    = self.layer_norm(torch.relu(self.cnn_proj(pooled)))
        return self.head_label(ctx), self.head_type(ctx), self.head_action(ctx)

    def info(self):
        p = sum(x.numel() for x in self.parameters())
        return {'params': p, 'size_mb': round(p*4/1024/1024, 2)}


print(f'SentinelNet v4.0 defined | Device: {DEVICE}')

In [ ]:
# Cell 4 — Train
from collections import Counter, defaultdict

MAX_LEN  = 300
BATCH    = 128
EPOCHS   = 40
PATIENCE = 8

# Stratified 85/15 split
cls_idx = defaultdict(list)
for i, s in enumerate(samples):
    cls_idx[LABEL2IDX[s['label']]].append(i)
train_idx, val_idx = [], []
for idxs in cls_idx.values():
    n = max(1, int(len(idxs) * 0.15))
    val_idx.extend(idxs[:n])
    train_idx.extend(idxs[n:])

tr_s = [samples[i] for i in train_idx]
vl_s = [samples[i] for i in val_idx]
print(f'Train: {len(tr_s):,} | Val: {len(vl_s):,}')

tokenizer = SentinelTokenizer(max_vocab=15000)
tokenizer.build_vocab([s['text'] for s in tr_s])

PIN = DEVICE.type == 'cuda'
NW  = 2 if DEVICE.type == 'cuda' else 0
tr_dl = DataLoader(ThreatDataset(tr_s, tokenizer, MAX_LEN), batch_size=BATCH, shuffle=True,  num_workers=NW, pin_memory=PIN)
vl_dl = DataLoader(ThreatDataset(vl_s, tokenizer, MAX_LEN), batch_size=BATCH, shuffle=False, num_workers=NW, pin_memory=PIN)

model = SentinelNet(tokenizer.vocab_size).to(DEVICE)
print(f'Params: {model.info()["params"]:,} | Size: {model.info()["size_mb"]} MB')

# Class weights
dist = Counter(LABEL2IDX[s['label']] for s in tr_s)
w = torch.tensor([len(tr_s)/(4*dist.get(i,1)) for i in range(4)], dtype=torch.float).to(DEVICE)

crit_label  = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
crit_type   = nn.CrossEntropyLoss(label_smoothing=0.05)
crit_action = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=3e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4, steps_per_epoch=len(tr_dl), epochs=EPOCHS, pct_start=0.2)

def f1_weighted(preds, labels):
    tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
    for p, l in zip(preds, labels):
        if p == l: tp[l] += 1
        else: fp[p] += 1; fn[l] += 1
    f1s, ws = [], []
    for c in range(4):
        pr = tp[c]/(tp[c]+fp[c]+1e-8)
        rc = tp[c]/(tp[c]+fn[c]+1e-8)
        f1s.append(2*pr*rc/(pr+rc+1e-8))
        ws.append(labels.count(c))
    return sum(f*w for f,w in zip(f1s,ws))/sum(ws)

best_f1, best_state, no_imp = 0.0, None, 0
print(f'Training on {DEVICE}...\n')

for epoch in range(1, EPOCHS+1):
    model.train()
    tl = 0
    for xb, yb_l, yb_t, yb_a in tr_dl:
        xb   = xb.to(DEVICE)
        yb_l = yb_l.to(DEVICE)
        yb_t = yb_t.to(DEVICE)
        yb_a = yb_a.to(DEVICE)
        optimizer.zero_grad()
        l_l, l_t, l_a = model(xb)
        loss = 0.6*crit_label(l_l,yb_l) + 0.2*crit_type(l_t,yb_t) + 0.2*crit_action(l_a,yb_a)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        tl += loss.item()
    tl /= len(tr_dl)

    model.eval()
    vl = 0; preds = []; lbls = []
    with torch.no_grad():
        for xb, yb_l, yb_t, yb_a in vl_dl:
            xb   = xb.to(DEVICE)
            yb_l = yb_l.to(DEVICE)
            yb_t = yb_t.to(DEVICE)
            yb_a = yb_a.to(DEVICE)
            l_l, l_t, l_a = model(xb)
            vl += (0.6*crit_label(l_l,yb_l) + 0.2*crit_type(l_t,yb_t) + 0.2*crit_action(l_a,yb_a)).item()
            preds.extend(l_l.argmax(1).cpu().tolist())
            lbls.extend(yb_l.cpu().tolist())
    vl  /= len(vl_dl)
    acc  = sum(p==l for p,l in zip(preds,lbls))/len(lbls)
    f1   = f1_weighted(preds, lbls)
    print(f'Epoch {epoch:2d} | train={tl:.4f} | val={vl:.4f} | acc={acc:.2%} | f1={f1:.4f}')

    if f1 > best_f1:
        best_f1   = f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_imp    = 0
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            print(f'Early stop @ epoch {epoch}')
            break

model.load_state_dict(best_state)
print(f'\nBest F1: {best_f1:.4f}')

In [ ]:
# Cell 5 — Save + Download
from google.colab import files

checkpoint = {
    'model_state':  model.state_dict(),
    'model_config': model.info(),
    'hyperparams': {
        'embed_dim':   128,
        'num_filters': 128,
        'tf_layers':   2,
        'dropout':     0.3,
        'max_len':     MAX_LEN,
    },
    'author':        'who_is_the_black_hat',
    'version':       '4.0',
    'github':        'https://github.com/Mrsultan7890/osints',
    'saved_at':      time.strftime('%Y-%m-%d %H:%M:%S'),
    'labels':        THREAT_LABELS,
    'threat_types':  THREAT_TYPES,
    'action_hints':  ACTION_HINTS,
    'best_f1':       best_f1,
    'trained_on':    str(DEVICE),
    'train_samples': len(tr_s),
    'total_samples': len(samples),
}
torch.save(checkpoint, 'sentinel_threat_net.pt')
tokenizer.save('sentinel_vocab.json')

print(f'Best F1      : {best_f1:.4f}')
print(f'Params       : {model.info()["params"]:,}')
print(f'Size         : {model.info()["size_mb"]} MB')
print(f'Train samples: {len(tr_s):,}')
print()
files.download('sentinel_threat_net.pt')
files.download('sentinel_vocab.json')
print('Downloaded!')
print()
print('Kali pe copy karo:')
print('cp ~/Downloads/sentinel_threat_net.pt /home/kali/osints/models/ml_engine/')
print('cp ~/Downloads/sentinel_vocab.json    /home/kali/osints/models/ml_engine/')

In [ ]:
# Cell 6 — Test
def predict(text):
    model.eval()
    ids = tokenizer.encode(text, MAX_LEN)
    x   = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        l_l, l_t, l_a = model(x)
        p_l = F.softmax(l_l, dim=-1)[0]
        p_t = F.softmax(l_t, dim=-1)[0]
        p_a = F.softmax(l_a, dim=-1)[0]
    return {
        'label':       IDX2LABEL[p_l.argmax().item()],
        'threat_type': IDX2TYPE[p_t.argmax().item()],
        'action_hint': IDX2HINT[p_a.argmax().item()],
        'confidence':  round(p_l.max().item(), 4),
    }

tests = [
    'nmap -sV -sC target.com open ports 22 80 443 3306 mysql running',
    'sqlmap found sql injection vulnerability login form database dumped',
    'ransomware encrypted all files bitcoin ransom demand critical infrastructure',
    'nikto scan found xss reflected vulnerability admin panel exposed',
    'gobuster found admin panel backup files exposed sensitive data',
    'normal website no vulnerabilities found ssl grade A headers ok',
]
print('=== Inference Test ===')
for t in tests:
    r = predict(t)
    print(f'[{r["label"]:8s}] type={r["threat_type"]:12s} action={r["action_hint"]:18s} conf={r["confidence"]:.2f}')
    print(f'  > {t[:70]}')
    print()

In [ ]:
# Cell 7 — Load All 3 Rounds Data
import json, glob, random
from collections import Counter

TOOLS = ['nmap','nikto','nuclei','sqlmap','gobuster','ffuf','amass','subfinder',
         'whatweb','wafw00f','sslscan','theHarvester','searchsploit','commix',
         'wpscan','enum4linux','hydra','masscan','hashcat','john']
TASKS = ['cmd_gen', 'chain_gen', 'report_gen']

def load_jsonl(path, max_samples=30000):
    samples = []
    seen = set()
    with open(path, encoding='utf-8', errors='replace') as f:
        for line in f:
            if len(samples) >= max_samples: break
            try:
                d = json.loads(line.strip())
                inp  = str(d.get('input','')).strip()
                out  = str(d.get('output','')).strip()
                task = d.get('task','')
                if not inp or not out or task not in TASKS: continue
                if len(inp) < 5 or len(out) < 2: continue
                key = inp[:60]
                if key in seen: continue
                seen.add(key)
                samples.append({
                    'input':  inp[:512],
                    'output': out[:256],
                    'task':   task,
                    'label':  d.get('label','HIGH'),
                    'tool':   d.get('tool',''),
                    'round':  d.get('round', 1),
                })
            except: pass
    return samples

r1_samples, r2_samples, r3_samples = [], [], []

for fname, var_name in [
    ('round1_basic.jsonl',  'r1'),
    ('round2_medium.jsonl', 'r2'),
    ('round3_high.jsonl',   'r3'),
]:
    found = glob.glob(f'/content/drive/MyDrive/**/{fname}', recursive=True)
    if not found: found = glob.glob(f'/content/drive/MyDrive/{fname}')
    if found:
        s = load_jsonl(found[0])
        if var_name == 'r1': r1_samples = s
        elif var_name == 'r2': r2_samples = s
        else: r3_samples = s
        print(f'✓ {fname}: {len(s):,} | tasks={dict(Counter(x["task"] for x in s))}')
    else:
        print(f'✗ {fname}: NOT FOUND in Drive')

print(f'\nTotal: R1={len(r1_samples):,} R2={len(r2_samples):,} R3={len(r3_samples):,}')


In [ ]:
# Cell 7.5 — BPE Tokenizer (train on all rounds combined)
!pip install tokenizers -q

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Split
from tokenizers.processors import TemplateProcessing
from tokenizers import Regex
import json, re

class BPESeqTokenizer:
    PAD_TOKEN = '<PAD>'; BOS_TOKEN = '<BOS>'
    EOS_TOKEN = '<EOS>'; UNK_TOKEN = '<UNK>'
    SPECIAL   = ['<PAD>', '<BOS>', '<EOS>', '<UNK>']

    def __init__(self):
        self._tok = None; self.vocab_size = 0
        self.PAD = 0; self.BOS = 1; self.EOS = 2; self.UNK = 3

    def train(self, texts, vocab_size=8000):
        corpus_path = '/tmp/bpe_corpus.txt'
        with open(corpus_path, 'w') as f:
            for t in texts:
                t = re.sub(r'([/\-_.,=])', r' \1 ', t.lower())
                t = re.sub(r'\s+', ' ', t).strip()
                if t: f.write(t + '\n')
        trainer = BpeTrainer(
            vocab_size=vocab_size, special_tokens=self.SPECIAL,
            min_frequency=2, show_progress=True,
            initial_alphabet=list('abcdefghijklmnopqrstuvwxyz0123456789-_./'),
        )
        tokenizer = Tokenizer(BPE(unk_token='<UNK>'))
        tokenizer.pre_tokenizer = Split(
            pattern=Regex(r'\s+|(?<=[a-z0-9])(?=[/\-_])|(?<=[/\-_])(?=[a-z0-9])'),
            behavior='removed', invert=False,
        )
        tokenizer.train([corpus_path], trainer)
        tokenizer.post_processor = TemplateProcessing(
            single=f'{self.BOS_TOKEN} $A {self.EOS_TOKEN}',
            special_tokens=[
                (self.BOS_TOKEN, tokenizer.token_to_id(self.BOS_TOKEN)),
                (self.EOS_TOKEN, tokenizer.token_to_id(self.EOS_TOKEN)),
            ],
        )
        self._tok = tokenizer
        self.vocab_size = tokenizer.get_vocab_size()
        self.PAD = tokenizer.token_to_id(self.PAD_TOKEN)
        self.BOS = tokenizer.token_to_id(self.BOS_TOKEN)
        self.EOS = tokenizer.token_to_id(self.EOS_TOKEN)
        self.UNK = tokenizer.token_to_id(self.UNK_TOKEN)
        print(f'BPE vocab: {self.vocab_size} | PAD={self.PAD} BOS={self.BOS} EOS={self.EOS}')

    def _norm(self, text):
        text = re.sub(r'([/\-_.,=])', r' \1 ', text.lower())
        return re.sub(r'\s+', ' ', text).strip()

    def encode_src(self, text, max_len=128):
        enc = self._tok.encode(self._norm(text))
        ids = [i for i in enc.ids if i not in (self.BOS, self.EOS)]
        ids = ids[:max_len]
        ids += [self.PAD] * (max_len - len(ids))
        return ids

    def encode_tgt(self, text, max_len=64):
        enc = self._tok.encode(self._norm(text))
        ids = enc.ids
        if not ids or ids[0] != self.BOS: ids = [self.BOS] + ids
        if len(ids) < max_len and ids[-1] != self.EOS: ids = ids + [self.EOS]
        ids = ids[:max_len]
        ids += [self.PAD] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        clean = [i for i in ids if i not in (self.PAD, self.BOS) and i != self.EOS]
        if self.EOS in ids: clean = clean[:ids.index(self.EOS)]
        text = self._tok.decode(clean)
        return re.sub(r'\s+([/\-_.,=])\s+', r'\1', text).strip()

    def save(self, path):
        self._tok.save(path.replace('.json', '_bpe.json'))
        json.dump({'type':'BPE','vocab_size':self.vocab_size,
                   'PAD':self.PAD,'BOS':self.BOS,'EOS':self.EOS,'UNK':self.UNK,
                   'bpe_file':path.replace('.json','_bpe.json')}, open(path,'w'))
        print(f'Saved: {path}')

# Train on all rounds combined
all_texts = ([s['input'] for s in r1_samples] + [s['output'] for s in r1_samples] +
             [s['input'] for s in r2_samples] + [s['output'] for s in r2_samples] +
             [s['input'] for s in r3_samples] + [s['output'] for s in r3_samples])
print(f'Training BPE on {len(all_texts):,} texts...')
seq_tok = BPESeqTokenizer()
seq_tok.train(all_texts, vocab_size=8000)
print('BPE tokenizer ready!')


In [ ]:
# Cell 8 — SentinelSeq2Seq v2.0 (CNN Encoder + Transformer Decoder)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import re, time, math, json
from collections import Counter as _Counter

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Tokenizer ──────────────────────────────────────────────────────────────
class SeqTokenizer:
    PAD, BOS, EOS, UNK = 0, 1, 2, 3
    SPECIAL = ['<PAD>','<BOS>','<EOS>','<UNK>']
    SEC_TERMS = {
        'nmap','sqlmap','nikto','nuclei','gobuster','ffuf','amass','subfinder',
        'whatweb','wafw00f','sslscan','theharvester','searchsploit','commix',
        'wpscan','enum4linux','hydra','masscan','hashcat','john','target',
        'scan','exploit','vulnerability','injection','xss','sqli','rce',
        'lfi','xxe','ssti','cors','jwt','breach','malware','phishing','apt',
        'recon','misconfig','critical','high','medium','low','patch','escalate',
        'cve','cvss','owasp','burp','metasploit','msfvenom','payload','shell',
        'reverse','bind','listener','exploit','privesc','lateral','persistence',
    }

    def __init__(self, max_vocab=10000):
        self.max_vocab = max_vocab
        self.w2i = {s: i for i, s in enumerate(self.SPECIAL)}
        self.i2w = {i: s for i, s in enumerate(self.SPECIAL)}
        self.vocab_size = len(self.SPECIAL)

    def _tok(self, text):
        # Path separators preserve karo — /usr/share/wordlists/rockyou.txt
        tokens = []
        for w in re.findall(r'[a-z0-9]+(?:[._/-][a-z0-9]+)*', text.lower()):
            if len(w) >= 1:
                tokens.append(w)
        return tokens

    def build_vocab(self, texts):
        cnt = _Counter()
        for t in texts: cnt.update(self._tok(t))
        for term in self.SEC_TERMS: cnt[term] = cnt.get(term,0) + 5000
        for w, _ in cnt.most_common(self.max_vocab - len(self.SPECIAL)):
            if w not in self.w2i:
                idx = len(self.w2i)
                self.w2i[w] = idx
                self.i2w[idx] = w
        self.vocab_size = len(self.w2i)
        print(f'Vocab: {self.vocab_size}')

    def encode_src(self, text, max_len=128):
        ids = [self.w2i.get(w, self.UNK) for w in self._tok(text)[:max_len]]
        ids += [self.PAD] * (max_len - len(ids))
        return ids

    def encode_tgt(self, text, max_len=64):
        ids = [self.BOS] + [self.w2i.get(w, self.UNK) for w in self._tok(text)[:max_len-2]] + [self.EOS]
        ids += [self.PAD] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        words = []
        for i in ids:
            if i == self.EOS: break
            if i in (self.PAD, self.BOS): continue
            w = self.i2w.get(i, '')
            if w: words.append(w)
        return ' '.join(words)

    def save(self, path):
        json.dump({'w2i': self.w2i,
                   'i2w': {str(k):v for k,v in self.i2w.items()},
                   'max_vocab': self.max_vocab}, open(path,'w'))

# ── Dataset ────────────────────────────────────────────────────────────────
class Seq2SeqDataset(Dataset):
    def __init__(self, samples, tok, src_len=128, tgt_len=64):
        self.src = [tok.encode_src(s['input'],  src_len) for s in samples]
        self.tgt = [tok.encode_tgt(s['output'], tgt_len) for s in samples]

    def __len__(self): return len(self.src)

    def __getitem__(self, i):
        return (torch.tensor(self.src[i], dtype=torch.long),
                torch.tensor(self.tgt[i], dtype=torch.long))

# ── Model ──────────────────────────────────────────────────────────────────
class SentinelSeq2Seq(nn.Module):
    """
    SentinelSeq2Seq v2.0
    Encoder : Embedding + CNN (k=3,5,7) — fast local pattern extraction
    Decoder : Transformer (4 heads, 3 layers) — better generation, less repetition
    ~35-45MB — size se koi problem nahi
    """
    VERSION = '2.0'
    AUTHOR  = 'who_is_the_black_hat'

    def __init__(self, vocab_size, embed_dim=256, num_filters=256,
                 nhead=4, dec_layers=3, ff_dim=512, dropout=0.1, pad_idx=0):
        super().__init__()
        self.embed_dim = embed_dim
        self.pad_idx   = pad_idx

        # Shared embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.emb_scale = math.sqrt(embed_dim)
        self.emb_drop  = nn.Dropout(dropout)

        # Encoder: CNN
        self.enc_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(embed_dim, num_filters, k, padding=k//2),
                nn.GELU(),
                nn.Dropout(dropout)
            ) for k in (3, 5, 7)
        ])
        self.enc_proj = nn.Linear(num_filters * 3, embed_dim)
        self.enc_norm = nn.LayerNorm(embed_dim)

        # Decoder: Transformer
        dec_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim, nhead=nhead,
            dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True,
            norm_first=True,   # Pre-LN — more stable training
        )
        self.decoder  = nn.TransformerDecoder(dec_layer, num_layers=dec_layers)
        self.out_proj = nn.Linear(embed_dim, vocab_size)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _pos_enc(self, x):
        B, L, D = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, D, 2, device=x.device).float() * (-math.log(10000.0) / D))
        pe  = torch.zeros(L, D, device=x.device)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:D//2])
        return x + pe.unsqueeze(0)

    def encode(self, src):
        # src: (B, L)
        emb = self.emb_drop(self.embedding(src) * self.emb_scale)  # (B, L, D)
        x   = emb.transpose(1, 2)  # (B, D, L)
        feats = torch.cat([c(x).transpose(1,2) for c in self.enc_convs], dim=-1)  # (B, L, 3F)
        enc_out = self.enc_norm(F.gelu(self.enc_proj(feats)))  # (B, L, D)
        return enc_out

    def forward(self, src, tgt):
        # src: (B, Ls) | tgt: (B, Lt)
        enc_out = self.encode(src)  # (B, Ls, D)

        # Decoder input: tgt[:-1] (teacher forcing)
        tgt_in  = tgt[:, :-1]  # (B, Lt-1)
        tgt_emb = self._pos_enc(
            self.emb_drop(self.embedding(tgt_in) * self.emb_scale)
        )  # (B, Lt-1, D)

        # Causal mask
        T = tgt_in.size(1)
        causal_mask = torch.triu(torch.ones(T, T, device=src.device), diagonal=1).bool()

        # Padding masks
        src_key_padding = (src == self.pad_idx)
        tgt_key_padding = (tgt_in == self.pad_idx)

        out = self.decoder(
            tgt_emb, enc_out,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=tgt_key_padding,
            memory_key_padding_mask=src_key_padding,
        )  # (B, Lt-1, D)

        return self.out_proj(out)  # (B, Lt-1, V)

    def generate(self, src, tok, max_len=64, temperature=0.7, top_k=50):
        """Top-k sampling — better diversity than greedy"""
        self.eval()
        with torch.no_grad():
            enc_out = self.encode(src)  # (1, Ls, D)
            generated = [tok.BOS]

            for _ in range(max_len):
                tgt_ids = torch.tensor([generated], dtype=torch.long, device=src.device)
                tgt_emb = self._pos_enc(
                    self.emb_drop(self.embedding(tgt_ids) * self.emb_scale)
                )
                T = tgt_ids.size(1)
                causal_mask = torch.triu(torch.ones(T, T, device=src.device), diagonal=1).bool()
                out = self.decoder(tgt_emb, enc_out, tgt_mask=causal_mask)
                logits = self.out_proj(out[:, -1, :]) / temperature  # (1, V)

                # Top-k sampling
                if top_k > 0:
                    vals, _ = torch.topk(logits, top_k)
                    logits[logits < vals[:, -1:]] = float('-inf')
                probs = F.softmax(logits, dim=-1)
                next_tok = torch.multinomial(probs, 1).item()

                if next_tok == tok.EOS: break
                generated.append(next_tok)

        return tok.decode(generated[1:])  # BOS skip

    def greedy(self, src, tok, max_len=64):
        """Greedy decoding — deterministic, chain_gen ke liye"""
        self.eval()
        with torch.no_grad():
            enc_out   = self.encode(src)
            generated = [tok.BOS]
            for _ in range(max_len):
                tgt_ids = torch.tensor([generated], dtype=torch.long, device=src.device)
                tgt_emb = self._pos_enc(
                    self.emb_drop(self.embedding(tgt_ids) * self.emb_scale)
                )
                T = tgt_ids.size(1)
                causal_mask = torch.triu(torch.ones(T, T, device=src.device), diagonal=1).bool()
                out    = self.decoder(tgt_emb, enc_out, tgt_mask=causal_mask)
                next_t = self.out_proj(out[:, -1, :]).argmax(-1).item()
                if next_t == tok.EOS: break
                generated.append(next_t)
        return tok.decode(generated[1:])

    def info(self):
        p = sum(x.numel() for x in self.parameters())
        return {'params': p, 'size_mb': round(p*4/1024/1024, 2),
                'version': self.VERSION}

print(f'SentinelSeq2Seq v2.0 defined | Device: {DEVICE}')
print('Architecture: CNN Encoder + Transformer Decoder (4 heads, 3 layers)')


In [ ]:
# Cell 9 — 3-Round Curriculum Training
import torch, torch.nn as nn, time, math
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

SRC_LEN = 128; TGT_LEN = 64
PIN = DEVICE.type == 'cuda'
NW  = 2 if DEVICE.type == 'cuda' else 0

def make_loaders(samples, batch_size):
    random.shuffle(samples)
    n_val = max(100, int(len(samples) * 0.1))
    tr, vl = samples[n_val:], samples[:n_val]
    tr_dl = DataLoader(Seq2SeqDataset(tr, seq_tok, SRC_LEN, TGT_LEN),
                       batch_size=batch_size, shuffle=True,  num_workers=NW, pin_memory=PIN)
    vl_dl = DataLoader(Seq2SeqDataset(vl, seq_tok, SRC_LEN, TGT_LEN),
                       batch_size=batch_size, shuffle=False, num_workers=NW, pin_memory=PIN)
    return tr_dl, vl_dl, tr, vl

def train_round(model, samples, round_num, lr, epochs, patience, batch_size=128):
    print(f'\n{'='*60}')
    print(f'ROUND {round_num} | {len(samples):,} samples | LR={lr} | Epochs={epochs}')
    print(f'{'='*60}')

    tr_dl, vl_dl, tr, vl = make_loaders(samples, batch_size)
    criterion = nn.CrossEntropyLoss(ignore_index=seq_tok.PAD, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(tr_dl),
        epochs=epochs, pct_start=0.1, anneal_strategy='cos')

    best_loss, best_state, no_imp = float('inf'), None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        tl = 0
        for src, tgt in tr_dl:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            optimizer.zero_grad()
            logits = model(src, tgt)
            B, T1, V = logits.shape
            loss = criterion(logits.reshape(B*T1, V), tgt[:, 1:].reshape(B*T1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step(); scheduler.step()
            tl += loss.item()
        tl /= len(tr_dl)

        model.eval(); vl = 0; acc = 0
        with torch.no_grad():
            for src, tgt in vl_dl:
                src, tgt = src.to(DEVICE), tgt.to(DEVICE)
                logits = model(src, tgt)
                B, T1, V = logits.shape
                vl  += criterion(logits.reshape(B*T1, V), tgt[:, 1:].reshape(B*T1)).item()
                pred = logits.argmax(-1); gold = tgt[:, 1:]
                mask = gold != seq_tok.PAD
                acc += (pred[mask] == gold[mask]).float().mean().item()
        vl /= len(vl_dl); acc /= len(vl_dl)
        print(f'  Epoch {epoch:2d} | train={tl:.4f} | val={vl:.4f} | acc={acc:.2%}')

        if vl < best_loss:
            best_loss = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'  Early stop @ epoch {epoch}')
                break

    model.load_state_dict(best_state)
    print(f'  Best val loss: {best_loss:.4f}')
    return best_loss

# ── Init model ────────────────────────────────────────────────────────────
seq_model = SentinelSeq2Seq(
    vocab_size=seq_tok.vocab_size, embed_dim=256, num_filters=256,
    nhead=4, dec_layers=3, ff_dim=512, dropout=0.1,
).to(DEVICE)
print(f'Model: {seq_model.info()["params"]:,} params | {seq_model.info()["size_mb"]} MB')

# ── Round 1: Linux basics — fresh training ────────────────────────────────
assert len(r1_samples) > 0, 'r1_samples empty — Cell 7 pehle run karo'
loss_r1 = train_round(seq_model, r1_samples,
                      round_num=1, lr=3e-4, epochs=15, patience=4)

# ── Round 2: Security tools — fine-tune ───────────────────────────────────
assert len(r2_samples) > 0, 'r2_samples empty — Cell 7 pehle run karo'
loss_r2 = train_round(seq_model, r2_samples,
                      round_num=2, lr=1e-4, epochs=15, patience=4)

# ── Round 3: Exploitation — fine-tune ────────────────────────────────────
assert len(r3_samples) > 0, 'r3_samples empty — Cell 7 pehle run karo'
loss_r3 = train_round(seq_model, r3_samples,
                      round_num=3, lr=5e-5, epochs=15, patience=4)

print(f'\n=== Curriculum Training Complete ===')
print(f'R1 loss: {loss_r1:.4f}')
print(f'R2 loss: {loss_r2:.4f}')
print(f'R3 loss: {loss_r3:.4f}')

# ── Quick inference test ──────────────────────────────────────────────────
print('\n=== Inference Test ===')
tests = [
    ('[SCAN_CONTEXT] list all files recursively [THREAT] LOW [TYPE] recon',                    'cmd_gen'),
    ('[SCAN_CONTEXT] web server port 80 open need vulnerability scan [THREAT] HIGH [TYPE] web_vuln', 'cmd_gen'),
    ('[SCAN_CONTEXT] sql injection suspected login form [THREAT] CRITICAL [TYPE] web_vuln',    'cmd_gen'),
    ('[CURRENT_TOOL] nmap [FINDING] open web port 80 443 [STATE] scan in progress',            'chain_gen'),
    ('[CURRENT_TOOL] nikto [FINDING] xss found admin panel [STATE] scan in progress',          'chain_gen'),
    ('[FINDINGS] ransomware encrypted files [SEVERITY] CRITICAL [TYPE] malware [ACTION] escalate', 'report_gen'),
]
for inp, task in tests:
    src_ids = torch.tensor([seq_tok.encode_src(inp, SRC_LEN)], dtype=torch.long).to(DEVICE)
    if task == 'chain_gen':
        out = seq_model.greedy(src_ids, seq_tok)
    else:
        out = seq_model.generate(src_ids, seq_tok, temperature=0.7, top_k=50)
    print(f'[{task}] {inp[:55]}')
    print(f'  → {out}')
    print()


In [ ]:
# Cell 10 — Save + Download SentinelSeq2Seq Final
from google.colab import files

checkpoint = {
    'model_state':  seq_model.state_dict(),
    'model_config': seq_model.info(),
    'hyperparams': {
        'vocab_size':  seq_tok.vocab_size,
        'embed_dim':   256,
        'num_filters': 256,
        'nhead':       4,
        'dec_layers':  3,
        'ff_dim':      512,
        'dropout':     0.1,
        'src_len':     SRC_LEN,
        'tgt_len':     TGT_LEN,
        'arch':        'CNN_Encoder_Transformer_Decoder',
        'curriculum':  '3-round',
    },
    'training': {
        'r1_loss': loss_r1, 'r2_loss': loss_r2, 'r3_loss': loss_r3,
        'r1_samples': len(r1_samples),
        'r2_samples': len(r2_samples),
        'r3_samples': len(r3_samples),
    },
    'tools':    TOOLS,
    'tasks':    TASKS,
    'author':   'who_is_the_black_hat',
    'version':  '2.0',
    'github':   'https://github.com/Mrsultan7890/osints',
    'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
}
torch.save(checkpoint, 'sentinel_seq2seq_final.pt')
seq_tok.save('sentinel_seq2seq_vocab.json')

print(f'Version  : {seq_model.info()["version"]}')
print(f'Size     : {seq_model.info()["size_mb"]} MB')
print(f'Params   : {seq_model.info()["params"]:,}')
print(f'R1 loss  : {loss_r1:.4f}')
print(f'R2 loss  : {loss_r2:.4f}')
print(f'R3 loss  : {loss_r3:.4f}')
print()
files.download('sentinel_seq2seq_final.pt')
files.download('sentinel_seq2seq_vocab.json')
files.download('sentinel_seq2seq_vocab_bpe.json')
print('Downloaded!')
print()
print('Kali pe copy karo:')
print('cp ~/Downloads/sentinel_seq2seq_final.pt   /home/kali/osints/models/ml_engine/sentinel_seq2seq.pt')
print('cp ~/Downloads/sentinel_seq2seq_vocab.json /home/kali/osints/models/ml_engine/')
print('cp ~/Downloads/sentinel_seq2seq_vocab_bpe.json /home/kali/osints/models/ml_engine/')
